# ATYDSE — Synchronous machine (SMIB) — adapted for pydae v1.x

Adapted from the legacy `dev/atydse_sm_profe.ipynb` to use the new monorepo API:

- `pydae.bps.BpsBuilder` (was `pydae.bmapu.bmapu_builder`)
- `pydae.core.Builder` + `pydae.core.Model` (was `grid.build('name')` + `import <name>; <name>.model()`)
- `pydae.ssa` is available for small-signal / eigenvalue analysis (Problema 3)

## Setup — install pydae and download data

In [1]:
from pydae.core import Builder, Model
from pydae.bps import BpsBuilder
from pydae.ssa import damp_report
import pydae
from plotly.subplots import make_subplots
import plotly.graph_objects as go
pydae.core.__version__

'1.0.0'

In [2]:
grid = BpsBuilder(str('milano4ord.hjson'))
grid.checker()
grid.uz_jacs = False
grid.construct('milano4ord')

bld = Builder(grid.sys_dict, target="cffi", sparse=False)
bld.build()

2026-04-17 12:29:41,125 Starting build pipeline for milano4ord (target=cffi, sparse=False)...
2026-04-17 12:29:41,127 Computing base symbolic Jacobians...


One generator must have K_delta > 0.0


2026-04-17 12:29:41,288 Building large global Jacobians...
2026-04-17 12:29:41,293 Converting Jacobians to target format (Sparse: False)
2026-04-17 12:29:41,328 Translating symbolic equations to C strings...
2026-04-17 12:29:41,457 End translating symbolic equations to C strings.
2026-04-17 12:29:41,458 [CFFI] Generating C code for system: milano4ord
2026-04-17 12:29:41,500 [CFFI] Compiling module 'milano4ord_cffi_dense' (backend: dense)
2026-04-17 12:29:44,445 [CFFI] Compilation successful. Python Module saved at: c:\Users\jmmau\workspace\pydae\examples\build\milano4ord_cffi_dense.cp313-win_amd64.pyd
2026-04-17 12:29:44,447 Build pipeline completed successfully!


In [3]:

model = Model('milano4ord')

In [4]:
# Initialization with steady-state mechanical power and voltage reference
model.ini({"p_m_1": 0.5, "v_ref_1": 1.0}, "xy_0.json")
model.report_x()
model.report_y()
model.report_z()

A = model.A_eval()          # reduced state matrix, shape (N_x, N_x)
df = damp_report(model, sparse=False, tol_part=0.2)
df.sort_values(['Damp'])

: 

In [21]:
    # Run 1 s at the initial operating point
model.ini({"p_m_1": 0.5}, "xy_0.json")

model.run(1.0, {})
model.run(10.0, {"p_m_1": 1.0}) # Step: increase mechanical power 

model.post()

In [22]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=model.Time, y=model.get_values("omega_1"), mode="lines", name=r"4ord: $\omega_1$"))
fig.update_layout(title="Short Circuit", xaxis_title="Time (s)", yaxis_title="Currents", hovermode="x")
fig.show()